# TF-IDF の仕組みと特徴を体験する

このノートブックでは、TF-IDFの以下の特徴を**実際の数値で**確認します。

| 特徴 | 内容 |
|------|------|
| ① ありふれた単語は低スコア | 「です」「ます」などはIDF値が低くなる |
| ② 珍しい単語は高スコア | 特定の文書にしか出ない単語は重要とみなされる |
| ③ 文脈・語順は無視される | 「好き」と「嫌い」が同じ扱いになってしまう |
| ④ 未知語は扱えない | 語彙リストにない単語はベクトルに反映されない |

---


## 0. セットアップ

In [ ]:
!apt-get install -y mecab libmecab-dev mecab-ipadic-utf8 fonts-ipafont > /dev/null 2>&1
!pip install mecab-python3 fugashi ipadic scikit-learn matplotlib seaborn pandas -q

import matplotlib, shutil
shutil.rmtree(matplotlib.get_cachedir(), ignore_errors=True)

print("✅ インストール完了")
print("⚠️  上のメニューから [ランタイム → セッションを再起動] してください")
print("   再起動後はセル1から実行してください（このセルの再実行は不要です）")


## 1. 共通設定（フォント・MeCab）

In [ ]:
import MeCab
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# 日本語フォント自動検出（フォントパスから直接登録）
jp_font_path = None
for f in fm.fontManager.ttflist:
    if any(kw in f.name for kw in ['IPA', 'Noto', 'Takao', 'VL Gothic']):
        jp_font_path = f.fname
        break

if jp_font_path:
    # フォントをパスで直接登録することで確実に反映させる
    fm.fontManager.addfont(jp_font_path)
    prop = fm.FontProperties(fname=jp_font_path)
    font_name = prop.get_name()
    matplotlib.rcParams['font.family'] = font_name
    matplotlib.rcParams['axes.unicode_minus'] = False
    print(f"使用フォント: {font_name} ({jp_font_path})")
else:
    print("⚠️ 日本語フォントが見つかりません。セッションを再起動してください。")

import subprocess, os

def find_mecab_dicdir():
    try:
        result = subprocess.run(["mecab-config", "--dicdir"],
                                capture_output=True, text=True)
        dicdir = result.stdout.strip()
        if dicdir:
            for subdir in ["ipadic", "mecab-ipadic-utf8", "ipadic-utf8"]:
                path = os.path.join(dicdir, subdir)
                if os.path.exists(path):
                    return path
            if os.path.exists(dicdir):
                return dicdir
    except FileNotFoundError:
        pass
    candidates = [
        "/usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ipadic-neologd",
        "/usr/lib/mecab/dic/mecab-ipadic-utf8",
        "/usr/lib/mecab/dic/ipadic",
        "/usr/local/lib/mecab/dic/ipadic",
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

dicdir = find_mecab_dicdir()
if dicdir:
    tagger = MeCab.Tagger(f"-d {dicdir}")
    print(f"辞書パス: {dicdir}")
else:
    tagger = MeCab.Tagger("-r /etc/mecabrc")
    print("mecabrc フォールバックで初期化")

def wakati(text):
    node = tagger.parseToNode(text)
    words = []
    while node:
        pos = node.feature.split(",")[0]
        if pos in ["名詞", "動詞", "形容詞"]:
            words.append(node.surface)
        node = node.next
    return " ".join(words)

test = wakati("日本語の形態素解析のテストです")
print(f"動作確認: {test}")
print("✅ 準備完了")


## 2. TF（単語の出現頻度）を確認する

まず、TF-IDFの「TF」部分だけを見てみましょう。  
TFは「その文書の中で、ある単語が何回出現するか」です。


In [ ]:
# サンプル文書（1つの文書でTFを確認）
doc = "寿司は美味しい。寿司が大好きだ。ラーメンも好きだ。"
words = wakati(doc).split()

print(f"文書: {doc}")
print(f"分かち書き後: {words}")
print(f"総単語数: {len(words)}")
print()

# 単語の出現回数をカウント
from collections import Counter
count = Counter(words)

print("【単語の出現回数】")
for word, cnt in count.most_common():
    tf = cnt / len(words)
    print(f"  '{word}': {cnt}回 → TF = {cnt}/{len(words)} = {tf:.3f}")


## 3. IDF（単語の珍しさ）を確認する

IDF は「全文書のうち何割の文書にその単語が出現するか」の逆数です。  
どの文書にも出てくる単語（助詞など）はIDFが低くなります。


In [ ]:
# 複数の文書でIDFを確認
documents = [
    "寿司は美味しい料理だ",
    "ラーメンは美味しい食べ物だ",
    "寿司とラーメンは人気の料理だ",
    "人工知能の技術が発展している",
    "機械学習は人工知能の一分野だ",
]

wakati_docs = [wakati(d) for d in documents]
print("【分かち書き後の文書】")
for i, (orig, wak) in enumerate(zip(documents, wakati_docs)):
    print(f"  文書{i+1}: {orig}")
    print(f"        → {wak}")
print()

# IDFを手動計算して確認
import math
all_words = set(" ".join(wakati_docs).split())
N = len(wakati_docs)

print("【IDF値の計算】（文書数 = {}件）".format(N))
print(f"  IDF = log(総文書数 / その単語を含む文書数)")
print()

idf_scores = {}
for word in sorted(all_words):
    df = sum(1 for d in wakati_docs if word in d.split())
    idf = math.log(N / df)
    idf_scores[word] = idf
    print(f"  '{word}': {df}件に出現 → IDF = log({N}/{df}) = {idf:.3f}")


## 4. TF-IDF = TF × IDF を確認する

TFとIDFを掛け合わせると、「その文書でよく出てきて、かつ他の文書にはあまり出てこない単語」が高スコアになります。


In [ ]:
# scikit-learnでTF-IDFベクトルを計算
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(wakati_docs)
feature_names = vectorizer.get_feature_names_out()

# 結果をDataFrameで表示
df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names,
    index=[f"文書{i+1}" for i in range(len(documents))]
)

print("【TF-IDFマトリクス】")
print("（0は「その文書にその単語が出現しない」ことを意味します）")
print()
print(df_tfidf.round(3).to_string())
print()

# 各文書の上位スコア単語を確認
print("【各文書で最も重要とみなされた単語（上位3語）】")
for i, doc in enumerate(documents):
    row = df_tfidf.iloc[i]
    top3 = row[row > 0].sort_values(ascending=False).head(3)
    print(f"  文書{i+1}「{doc}」")
    for word, score in top3.items():
        print(f"    → '{word}': {score:.3f}")


## 5. 特徴①：ありふれた単語は低スコアになる

「美味しい」はほぼすべての料理文書に出現するためIDFが低く、TF-IDFスコアも低くなります。


In [ ]:
# ヒートマップで可視化
fig, ax = plt.subplots(figsize=(12, 4))

sns.heatmap(
    df_tfidf,
    ax=ax,
    annot=True, fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "TF-IDFスコア"}
)
ax.set_title("TF-IDFマトリクス：ありふれた単語は低スコア", fontsize=13, fontweight="bold")
ax.set_xlabel("単語")
ax.set_ylabel("文書")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 「美味しい」のスコアを確認
if "美味しい" in df_tfidf.columns:
    print("「美味しい」のTF-IDFスコア:")
    for i, score in enumerate(df_tfidf["美味しい"]):
        print(f"  文書{i+1}: {score:.3f}")
    print("→ 複数の文書に出現するため、スコアが低くなっています")


## 6. 特徴②：文脈・感情が無視される

TF-IDFは単語の「出現」しか見ないため、「好き」と「嫌い」を区別できません。  
意味が真逆な文章でも、似たベクトルになってしまいます。


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# 意味が正反対の文章ペア
contrast_docs = [
    "寿司が大好きだ",       # ポジティブ
    "寿司が大嫌いだ",       # ネガティブ（意味は逆）
    "ラーメンが食べたい",   # 全く別のトピック
]

wakati_contrast = [wakati(d) for d in contrast_docs]

print("【分かち書き後】")
for orig, wak in zip(contrast_docs, wakati_contrast):
    print(f"  「{orig}」→ {wak}")
print()

vec = TfidfVectorizer()
matrix = vec.fit_transform(wakati_contrast).toarray()
sim = cosine_similarity(matrix)

print("【コサイン類似度】")
print("  （1.0 = 完全に同じ、0.0 = 全く異なる）")
print()
labels = [f"「{d}」" for d in contrast_docs]
df_sim = pd.DataFrame(sim, index=labels, columns=labels)
print(df_sim.round(3).to_string())
print()
print("⚠️  「寿司が大好きだ」と「寿司が大嫌いだ」の類似度が高い！")
print("   TF-IDFは感情・文脈を無視するため、意味が逆でも似たベクトルになります。")


## 7. 特徴③：未知語はベクトルに反映されない

`fit` はモデルの学習ではなく、**文書群に含まれる単語から語彙リストを構築する処理**です。  
語彙リストに存在しない単語（新語・造語・専門用語など）は変換する列が存在しないため、ベクトルがすべて0になります。


In [ ]:
# 語彙リスト構築用の文書
train_docs = [
    "人工知能の技術が発展している",
    "機械学習はデータを学習する",
]

# 変換対象の文書（新語を含む）
test_docs = [
    "生成AIとRAGを組み合わせたシステム",   # 「生成AI」「RAG」は未知語
    "機械学習でデータを分析する",            # 既知の単語のみ
]

wakati_train = [wakati(d) for d in train_docs]
wakati_test  = [wakati(d) for d in test_docs]

vec = TfidfVectorizer()
vec.fit(wakati_train)  # この文書群から語彙リストを構築する

print("【構築された語彙リスト】")
print(f"  {sorted(vec.vocabulary_.keys())}")
print()

# テストデータを変換
test_matrix = vec.transform(wakati_test).toarray()
feature_names = vec.get_feature_names_out()

print("【テストデータのTF-IDFベクトル】")
for doc, orig, vec_row in zip(wakati_test, test_docs, test_matrix):
    nonzero = {feature_names[i]: round(v, 3) for i, v in enumerate(vec_row) if v > 0}
    print(f"  「{orig}」")
    print(f"    分かち書き: {doc}")
    if nonzero:
        print(f"    ベクトル（非ゼロ要素）: {nonzero}")
    else:
        print(f"    ベクトル: すべて0（既知の単語が1つもない）")
    print()

print("⚠️  「生成AI」「RAG」は学習データにないため、ベクトルに反映されません。")


## 8. スパースベクトルの可視化

TF-IDFのベクトルは「ほとんどが0」のスパース（疎）なベクトルです。  
BERTの密（Dense）なベクトルと比較してみましょう。


In [ ]:
# より多くの文書でスパース性を確認
many_docs = [
    "寿司は美味しい料理だ",
    "ラーメンは人気の食べ物だ",
    "人工知能の技術が発展している",
    "機械学習はデータから学習する",
    "野球の試合で日本代表が勝利した",
    "サッカーのワールドカップが開催された",
]

wakati_many = [wakati(d) for d in many_docs]
vec_many = TfidfVectorizer()
matrix_many = vec_many.fit_transform(wakati_many).toarray()

total_elements = matrix_many.size
nonzero_elements = np.count_nonzero(matrix_many)
sparsity = 1 - nonzero_elements / total_elements

dim = matrix_many.shape[1]
print(f"語彙数（ベクトルの次元数）: {dim}")
print(f"文書数: {matrix_many.shape[0]}")
print(f"全要素数: {total_elements}")
print(f"非ゼロ要素数: {nonzero_elements}")
print(f"スパース率: {sparsity:.1%}")
print()

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TF-IDFのスパースベクトル
axes[0].spy(matrix_many, markersize=8, color="#e74c3c")
tfidf_title = "TF-IDF ベクトル ({}次元) スパース率: {}".format(dim, f"{sparsity:.1%}")
axes[0].set_title(tfidf_title, fontsize=12, fontweight="bold")
axes[0].set_xlabel("次元（単語のインデックス）")
axes[0].set_ylabel("文書")
axes[0].set_yticks(range(len(many_docs)))
axes[0].set_yticklabels(["文書" + str(i+1) for i in range(len(many_docs))], fontsize=9)

# BERTの密なベクトルのイメージ（ランダムで代替表示）
np.random.seed(42)
dense_mock = np.random.randn(len(many_docs), 768)
axes[1].imshow(dense_mock, cmap="Blues", aspect="auto", vmin=-3, vmax=3)
axes[1].set_title("BERT ベクトル (768次元)
スパース率: 0%（すべての次元に値がある）", fontsize=12, fontweight="bold")
axes[1].set_xlabel("次元（0〜767）")
axes[1].set_ylabel("文書")
axes[1].set_yticks(range(len(many_docs)))
axes[1].set_yticklabels(["文書" + str(i+1) for i in range(len(many_docs))], fontsize=9)

plt.suptitle("スパースベクトル（TF-IDF）vs 密なベクトル（BERT）", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 9. まとめ

### TF-IDFの特徴

| 特徴 | 内容 | 確認できたこと |
|------|------|--------------|
| ✅ ありふれた単語は低スコア | IDFが低くなる | 「美味しい」など複数文書に出る単語はスコアが低い |
| ✅ 珍しい単語は高スコア | IDFが高くなる | 特定文書にしか出ない単語が重要とみなされる |
| ⚠️ 文脈・感情が無視される | 単語の出現だけを見る | 「好き」と「嫌い」が区別できない |
| ⚠️ 未知語は扱えない | 語彙外の単語は0になる | 「生成AI」「RAG」が無視された |
| ⚠️ スパースベクトル | ほとんどの次元が0 | 語彙数が増えると次元数も増大する |

### BERTとの使い分け

```
TF-IDF が向いている場面:
  - キーワード完全一致が重要な検索
  - 大量文書の高速処理
  - 計算リソースが限られている環境

BERT が向いている場面:
  - 意味的な類似検索（RAGなど）
  - 感情分析・感情を含む分類
  - 新語・造語・専門用語を扱う場合
```
